In [1]:
import gc
import os
import pickle
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

def ram_gb():
    return psutil.virtual_memory().available / 1e9

DATASETS = [
    {"label": "medium", "accounts_path": r"../data/aml/LI-Medium_accounts.csv", "trans_path": r"../data/aml/LI-Medium_Trans.csv"},
    {"label": "large",  "accounts_path": r"../data/aml/LI-Large_accounts.csv",  "trans_path": r"../data/aml/LI-Large_Trans.csv"},
]
ENTITY_TYPES_NEEDED = ["Individual", "Sole Proprietorship"]

def load_raw_accounts(accounts_path):
    a = pd.read_csv(
        accounts_path,
        dtype={"Bank Name": "category", "Bank ID": "int32", "Account Number": "string",
               "Entity ID": "category", "Entity Name": "string"},
    )
    a["Entity Type"] = a["Entity Name"].str.extract(r"^([A-Za-z ]+) #").astype("category")
    return a

def prepare_accounts(raw_accounts, source_label):
    a = raw_accounts.copy()
    foreign_pattern = r"^[A-Za-z ]+ Bank #\d+$"
    a["Is_Foreign_Bank"] = a["Bank Name"].str.match(foreign_pattern)
    us_bank_ids = set(a.loc[~a["Is_Foreign_Bank"], "Bank ID"].unique())
    a_us = a[a["Bank ID"].isin(us_bank_ids) & a["Entity Type"].isin(ENTITY_TYPES_NEEDED)].copy()
    a_us["Source_Dataset"] = source_label
    a_us["Account Number Raw"] = a_us["Account Number"]
    a_us["Account Number"] = source_label + "_" + a_us["Account Number Raw"].astype(str)
    a_us["Entity ID"] = source_label + "_" + a_us["Entity ID"].astype(str)
    return a_us, us_bank_ids

accounts_us_parts, us_bank_ids_by_source, raw2global_by_source = [], {}, {}
for ds in DATASETS:
    label = ds["label"]
    raw = load_raw_accounts(ds["accounts_path"])
    a_us, us_bank_ids = prepare_accounts(raw, label)
    accounts_us_parts.append(a_us)
    us_bank_ids_by_source[label] = us_bank_ids
    raw2global_by_source[label] = dict(zip(a_us["Account Number Raw"], a_us["Account Number"]))
    print(f"{label}: {a_us.shape[0]:,} US accounts, {a_us['Entity ID'].nunique():,} entities")

accounts_us = pd.concat(accounts_us_parts, ignore_index=True)
acct2entity = dict(zip(accounts_us["Account Number"], accounts_us["Entity ID"]))
acct2etype  = dict(zip(accounts_us["Account Number"], accounts_us["Entity Type"]))
print("\nCombined accounts:", accounts_us.shape)

Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Asus\\.matplotlib\\fontlist-v390.json.matplotlib-lock'


medium: 267,419 US accounts, 93,746 entities
large: 271,380 US accounts, 93,870 entities

Combined accounts: (538799, 9)


In [2]:
CHUNKSIZE = 2_000_000
usecols = ["Timestamp", "From Bank", "Account", "To Bank", "Account.1",
           "Amount Received", "Receiving Currency", "Amount Paid",
           "Payment Currency", "Payment Format", "Is Laundering"]
dtype_map = {
    "From Bank": "int32", "To Bank": "int32", "Amount Received": "float32", "Amount Paid": "float32",
    "Receiving Currency": "category", "Payment Currency": "category",
    "Payment Format": "category", "Is Laundering": "int8",
}

sum_log_amount = Counter(); sumsq_log_amount = Counter(); n_amount = Counter()
format_counts_by_type = {t: Counter() for t in ENTITY_TYPES_NEEDED}
laundering_sum_by_type = Counter(); laundering_n_by_type = Counter()
role_counts_by_type = {t: Counter() for t in ENTITY_TYPES_NEEDED}

for ds in DATASETS:
    label = ds["label"]
    us_bank_ids = us_bank_ids_by_source[label]
    raw2global = raw2global_by_source[label]

    print(f"\n=== Streaming {label} ===")
    reader = pd.read_csv(ds["trans_path"], usecols=usecols, dtype=dtype_map, chunksize=CHUNKSIZE)
    for i, chunk in enumerate(reader):
        chunk = chunk[chunk["From Bank"].isin(us_bank_ids) & chunk["To Bank"].isin(us_bank_ids)]
        if len(chunk) == 0:
            continue

        acct_global = chunk["Account"].map(raw2global)
        acct1_global = chunk["Account.1"].map(raw2global)
        sender_etype = acct_global.map(acct2etype)
        receiver_etype = acct1_global.map(acct2etype)

        recv_amt = chunk["Amount Received"].astype("float64")
        valid_recv = receiver_etype.notna() & (recv_amt > 0)
        if valid_recv.any():
            log_amt = np.log(recv_amt[valid_recv])
            grp = receiver_etype[valid_recv]
            sum_log_amount.update(log_amt.groupby(grp, observed=True).sum().to_dict())
            sumsq_log_amount.update((log_amt ** 2).groupby(grp, observed=True).sum().to_dict())
            n_amount.update(grp.value_counts().to_dict())

        for etype in ENTITY_TYPES_NEEDED:
            s_mask = sender_etype == etype
            r_mask = receiver_etype == etype
            if s_mask.any():
                format_counts_by_type[etype].update(chunk.loc[s_mask, "Payment Format"].value_counts().to_dict())
                laundering_sum_by_type[etype] += chunk.loc[s_mask, "Is Laundering"].sum()
                laundering_n_by_type[etype] += s_mask.sum()
                role_counts_by_type[etype]["sender"] += s_mask.sum()
            if r_mask.any():
                role_counts_by_type[etype]["receiver"] += r_mask.sum()

        del chunk, acct_global, acct1_global, sender_etype, receiver_etype, recv_amt
        if i % 3 == 0:
            gc.collect()
            print(f"  [{label}] chunk {i} | RAM {ram_gb():.2f} GB")

# ---- fitted distributions ----
amount_models = {}
for etype in ENTITY_TYPES_NEEDED:
    n = n_amount[etype]
    mean_log = sum_log_amount[etype] / n
    var_log = (sumsq_log_amount[etype] / n) - mean_log ** 2
    amount_models[etype] = {"mean_log": mean_log, "std_log": np.sqrt(max(var_log, 1e-6))}

format_probs_by_type = {
    etype: {k: v / sum(format_counts_by_type[etype].values()) for k, v in format_counts_by_type[etype].items()}
    for etype in ENTITY_TYPES_NEEDED
}
laundering_rate_by_type = {
    etype: laundering_sum_by_type[etype] / laundering_n_by_type[etype] for etype in ENTITY_TYPES_NEEDED
}
role_probs_by_type = {
    etype: {
        "sender": role_counts_by_type[etype]["sender"] / sum(role_counts_by_type[etype].values()),
        "receiver": role_counts_by_type[etype]["receiver"] / sum(role_counts_by_type[etype].values()),
    }
    for etype in ENTITY_TYPES_NEEDED
}

print("Amount models:", amount_models)
print("Laundering rate:", laundering_rate_by_type)
print("Role split:", role_probs_by_type)


=== Streaming medium ===
  [medium] chunk 0 | RAM 2.10 GB
  [medium] chunk 3 | RAM 2.01 GB
  [medium] chunk 6 | RAM 2.01 GB
  [medium] chunk 9 | RAM 2.00 GB
  [medium] chunk 12 | RAM 1.97 GB
  [medium] chunk 15 | RAM 1.95 GB

=== Streaming large ===
  [large] chunk 0 | RAM 2.00 GB
  [large] chunk 3 | RAM 1.94 GB
  [large] chunk 6 | RAM 2.04 GB
  [large] chunk 9 | RAM 2.12 GB
  [large] chunk 12 | RAM 2.07 GB
  [large] chunk 15 | RAM 2.07 GB
  [large] chunk 18 | RAM 2.09 GB
  [large] chunk 21 | RAM 1.95 GB
  [large] chunk 24 | RAM 1.97 GB
  [large] chunk 27 | RAM 1.90 GB
  [large] chunk 30 | RAM 1.96 GB
  [large] chunk 33 | RAM 1.96 GB
  [large] chunk 36 | RAM 1.86 GB
  [large] chunk 39 | RAM 1.81 GB
  [large] chunk 42 | RAM 1.80 GB
  [large] chunk 45 | RAM 2.00 GB
  [large] chunk 48 | RAM 1.97 GB
  [large] chunk 51 | RAM 1.80 GB
  [large] chunk 54 | RAM 1.87 GB
  [large] chunk 57 | RAM 1.14 GB
  [large] chunk 60 | RAM 1.24 GB
  [large] chunk 63 | RAM 1.31 GB
  [large] chunk 66 | RAM 1.

In [3]:
# ---- reference data: monthly transaction volume by income bracket (2022 diary of consumer payment choice) ----
MONTHLY_TXN_RANGES = {
    "Low":    (25, 28),    # under $25,000/year
    "Middle": (38, 44),    # $25,000 - $100,000/year
    "High":   (55, 65),    # over $100,000/year
}
POPULATION_WEIGHTS = {"Low": 0.23, "Middle": 0.44, "High": 0.33}  # Low+High reported; Middle = remainder

# assumed numeric bounds per bracket, used only to sample a concrete target income
# for customers with no known Annual_Income (i.e. the test split)
INCOME_BRACKET_BOUNDS = {"Low": (5_000, 25_000), "Middle": (25_000, 100_000), "High": (100_000, 300_000)}

def get_income_bracket(income):
    if pd.isna(income) or income <= 0:
        return None
    if income < 25_000:
        return "Low"
    elif income < 100_000:
        return "Middle"
    else:
        return "High"

N_ACCOUNTS_RANGE = range(1, 13)          # 1 to 12
ENTITIES_PER_BUCKET = 500                 # per (n_accounts, entity_type, bracket) combination
SIMULATION_DAYS = 365

rng = np.random.default_rng(42)

def generate_entity_transactions(entity_id, entity_type, bracket, rng):
    """Generate one entity's annual transaction set, volume calibrated to its income bracket,
       amount shape drawn from real fitted distribution, then rescaled so the receiver-side
       sum matches a bracket-consistent annual income target."""
    monthly_txn = rng.uniform(*MONTHLY_TXN_RANGES[bracket])
    n = max(int(round(monthly_txn * 12)), 1)

    amt_model = amount_models[entity_type]
    raw_amounts = rng.lognormal(mean=amt_model["mean_log"], sigma=amt_model["std_log"], size=n)

    role_p = role_probs_by_type[entity_type]
    roles = rng.choice(["sender", "receiver"], size=n, p=[role_p["sender"], role_p["receiver"]])

    # rescale so receiver-side sum lands on a bracket-consistent target annual income
    lo, hi = INCOME_BRACKET_BOUNDS[bracket]
    target_income = rng.uniform(lo, hi)
    receiver_sum_raw = raw_amounts[roles == "receiver"].sum()
    scale = target_income / receiver_sum_raw if receiver_sum_raw > 0 else 1.0
    amounts = raw_amounts * scale

    fmt_names, fmt_p = zip(*format_probs_by_type[entity_type].items())
    formats = rng.choice(fmt_names, size=n, p=fmt_p)

    days_offset = rng.integers(0, SIMULATION_DAYS, size=n)
    hours = rng.integers(0, 24, size=n)
    minutes = rng.integers(0, 60, size=n)
    base_date = pd.Timestamp("2023-01-01")
    timestamps = base_date + pd.to_timedelta(days_offset, unit="D") \
                            + pd.to_timedelta(hours, unit="h") + pd.to_timedelta(minutes, unit="m")

    is_laundering = rng.choice([0, 1], size=n,
                                p=[1 - laundering_rate_by_type[entity_type], laundering_rate_by_type[entity_type]])

    from_bank = abs(hash(entity_id)) % (2**30 - 1)
    account_id = f"SIM_{entity_id}"
    to_banks = rng.integers(0, 2**30 - 1, size=n)
    counterpart_accounts = [f"SIM_EXT_{rng.integers(0, 10**9)}" for _ in range(n)]

    return pd.DataFrame({
        "Timestamp": timestamps, "From Bank": from_bank, "Account": account_id,
        "To Bank": to_banks, "Account.1": counterpart_accounts,
        "Amount Received": np.round(amounts, 2), "Receiving Currency": "US Dollar",
        "Amount Paid": np.round(amounts, 2), "Payment Currency": "US Dollar",
        "Payment Format": formats, "Is Laundering": is_laundering, "_role": roles,
    })

# ---- build the pool: entities for every (n_accounts, entity_type, income_bracket) bucket ----
synthetic_entity_pool = {}          # (n_accounts, entity_type, bracket) -> [entity_id, ...]
synthetic_entity_transactions = {}
entity_counter = 0

for etype in ENTITY_TYPES_NEEDED:
    for n_acc in N_ACCOUNTS_RANGE:
        for bracket in ["Low", "Middle", "High"]:
            ids_this_bucket = []
            for _ in range(ENTITIES_PER_BUCKET):
                eid = f"SYN_{etype[:4]}_{n_acc}_{bracket}_{entity_counter}"
                entity_counter += 1
                txns = generate_entity_transactions(eid, etype, bracket, rng)
                synthetic_entity_transactions[eid] = txns
                ids_this_bucket.append(eid)
            synthetic_entity_pool[(n_acc, etype, bracket)] = ids_this_bucket

print(f"Generated {len(synthetic_entity_transactions):,} synthetic entities "
      f"across {len(synthetic_entity_pool)} buckets "
      f"({len(N_ACCOUNTS_RANGE)} account values × {len(ENTITY_TYPES_NEEDED)} types × 3 brackets)")

Generated 36,000 synthetic entities across 72 buckets (12 account values × 2 types × 3 brackets)


In [4]:
def map_occupation(occ):
    if occ == "Entrepreneur":
        return "Sole Proprietorship"
    elif occ == "Unknown":
        return None
    else:
        return "Individual"

def get_most_frequent(series):
    mode_vals = series.mode()
    return mode_vals.iloc[0] if not mode_vals.empty else "Unknown"

def build_customer_profile(raw_df):
    has_income = "Annual_Income" in raw_df.columns
    agg_cols = ["Occupation", "Num_Bank_Accounts"]
    if has_income:
        agg_cols.append("Annual_Income")

    profile = raw_df.groupby("Customer_ID")[agg_cols].agg(get_most_frequent).reset_index()
    if not has_income:
        profile["Annual_Income"] = np.nan

    profile["Num_Bank_Accounts"] = profile["Num_Bank_Accounts"].round().astype(int).clip(1, 12)
    profile["Assumed_Entity_Type"] = profile["Occupation"].apply(map_occupation)

    entity_type_probs = pd.Series({"Individual": 0.6, "Sole Proprietorship": 0.4})  # or derive from df_entity counts
    rng_unknown = np.random.default_rng(11)
    mask = profile["Assumed_Entity_Type"].isna()
    profile.loc[mask, "Assumed_Entity_Type"] = rng_unknown.choice(
        entity_type_probs.index, size=mask.sum(), p=entity_type_probs.values
    )

    # ---- income bracket: known income -> derived bracket; unknown -> sampled by population weights ----
    profile["Income_Bracket"] = profile["Annual_Income"].apply(get_income_bracket)
    rng_bracket = np.random.default_rng(13)
    mask_b = profile["Income_Bracket"].isna()
    profile.loc[mask_b, "Income_Bracket"] = rng_bracket.choice(
        list(POPULATION_WEIGHTS.keys()), size=mask_b.sum(), p=list(POPULATION_WEIGHTS.values())
    )
    return profile

In [6]:
def assign_and_build_transactions(profile, synthetic_entity_pool, synthetic_entity_transactions, seed=2024):
    rng = np.random.default_rng(seed)
    used_per_bucket = {}
    assigned_eids = []
    frames = []
    reuse_count = 0

    for row in profile.itertuples(index=False):
        key = (row.Num_Bank_Accounts, row.Assumed_Entity_Type, row.Income_Bracket)
        candidates = synthetic_entity_pool.get(key, [])
        if not candidates:
            assigned_eids.append(None)
            continue

        used = used_per_bucket.setdefault(key, set())
        available = [c for c in candidates if c not in used]
        if not available:
            available = candidates
            reuse_count += 1

        eid = rng.choice(available)   # random assignment, as requested
        used.add(eid)
        assigned_eids.append(eid)

        txns = synthetic_entity_transactions[eid].copy()

        # ---- fine-scale rescale: match the customer's EXACT annual income when known ----
        if pd.notna(row.Annual_Income) and row.Annual_Income > 0:
            receiver_sum = txns.loc[txns["_role"] == "receiver", "Amount Received"].sum()
            if receiver_sum > 0:
                scale = row.Annual_Income / receiver_sum
                txns["Amount Received"] = (txns["Amount Received"] * scale).round(2)
                txns["Amount Paid"] = (txns["Amount Paid"] * scale).round(2)

        txns["Customer_ID"] = row.Customer_ID
        txns["Source_Entity_ID"] = eid
        txns["Assumed_Entity_Type"] = row.Assumed_Entity_Type
        txns["Income_Bracket"] = row.Income_Bracket
        frames.append(txns)

    print(f"  Entities reused due to bucket exhaustion: {reuse_count:,} / {len(profile):,} customers")
    profile = profile.copy()
    profile["Assigned_Entity_ID"] = assigned_eids
    txns_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    return profile, txns_df

def estimate_income(cust_txn_df):
    def agg(g):
        n = len(g)
        received = g.loc[g["_role"] == "receiver", "Amount Received"].sum()
        return pd.Series({"N_Transactions": n, "Sum_Received": received, "Estimated_Annual_Income": received})
    return cust_txn_df.groupby("Customer_ID").apply(agg).reset_index()

def process_split(raw_df, seed=2024):
    profile = build_customer_profile(raw_df)
    profile, txns_df = assign_and_build_transactions(profile, synthetic_entity_pool, synthetic_entity_transactions, seed)
    income_df = estimate_income(txns_df)
    profile = profile.drop(columns=[c for c in ["N_Transactions", "Sum_Received", "Estimated_Annual_Income"]
                                     if c in profile.columns])
    profile = profile.merge(income_df, on="Customer_ID", how="left")
    return profile, txns_df

# ---- train ----
train_df = pd.read_csv("../data/cleaned_credit/cleaned_train.csv", low_memory=False)
print("Processing train split...")
customer_profile, customer_transactions = process_split(train_df, seed=2024)
customer_profile.to_csv("../data/simulated/customer_profile_train.csv", index=False)
customer_transactions.to_csv("../data/simulated/customer_transactions_train.csv", index=False)
print(customer_profile.shape, customer_transactions.shape)

# ---- test ----
test_df = pd.read_csv("../data/cleaned_credit/cleaned_test.csv", low_memory=False)
print("\nProcessing test split...")
test_profile, test_transactions = process_split(test_df, seed=4242)
test_profile.to_csv("../data/simulated/customer_profile_test.csv", index=False)
test_transactions.to_csv("../data/simulated/customer_transactions_test.csv", index=False)
print(test_profile.shape, test_transactions.shape)

Processing train split...
  Entities reused due to bucket exhaustion: 2,224 / 12,500 customers


C:\Users\Asus\AppData\Local\Temp\ipykernel_7952\320859566.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cust_txn_df.groupby("Customer_ID").apply(agg).reset_index()


(12500, 10) (5779573, 16)

Processing test split...
  Entities reused due to bucket exhaustion: 2,215 / 12,500 customers


C:\Users\Asus\AppData\Local\Temp\ipykernel_7952\320859566.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return cust_txn_df.groupby("Customer_ID").apply(agg).reset_index()


(12500, 10) (5780796, 16)


In [7]:
def check_uniqueness(profile, label="train"):
    print(f"\n================ UNIQUENESS CHECK: {label} ================")
    valid = profile.dropna(subset=["Assigned_Entity_ID"])
    dup = valid["Assigned_Entity_ID"].value_counts()
    print(f"Customers: {len(valid):,} | Unique entities used: {valid['Assigned_Entity_ID'].nunique():,}")
    print(f"Entities reused: {(dup > 1).sum():,}")

def check_income_match(profile, label="train"):
    print(f"\n================ INCOME MATCH CHECK: {label} ================")
    has_income = profile["Annual_Income"].notna() & (profile["Annual_Income"] > 0)
    if has_income.any():
        err = (profile.loc[has_income, "Estimated_Annual_Income"] - profile.loc[has_income, "Annual_Income"]).abs()
        rel_err = err / profile.loc[has_income, "Annual_Income"]
        print(f"Income-known customers: {has_income.sum():,}")
        print(f"Median relative income error: {rel_err.median():.4%}  (should be ~0%, exact rescale applied)")
    else:
        print("No known Annual_Income in this split (expected for test).")

def check_txn_volume_by_bracket(profile, label="train"):
    print(f"\n================ TXN VOLUME BY BRACKET CHECK: {label} ================")
    profile = profile.copy()
    profile["Monthly_Txn"] = profile["N_Transactions"] / 12
    print(profile.groupby("Income_Bracket")["Monthly_Txn"].describe())
    print("\nReference ranges:", MONTHLY_TXN_RANGES)

def spot_check(profile, n=8, seed=99, label="train"):
    print(f"\n================ SPOT CHECK: {label} ================")
    valid = profile.dropna(subset=["Assigned_Entity_ID"])
    for row in valid.sample(min(n, len(valid)), random_state=seed).itertuples(index=False):
        print(f"\nCustomer {row.Customer_ID} | Occupation={row.Occupation} | "
              f"Assumed_Entity_Type={row.Assumed_Entity_Type} | Num_Bank_Accounts={row.Num_Bank_Accounts} | "
              f"Income_Bracket={row.Income_Bracket}")
        print(f"  Assigned_Entity_ID={row.Assigned_Entity_ID} | N_Transactions={row.N_Transactions} | "
              f"Annual_Income={row.Annual_Income} | Estimated_Annual_Income={row.Estimated_Annual_Income:.2f}")

check_uniqueness(customer_profile, "train")
check_income_match(customer_profile, "train")
check_txn_volume_by_bracket(customer_profile, "train")
spot_check(customer_profile, label="train")

check_uniqueness(test_profile, "test")
check_txn_volume_by_bracket(test_profile, "test")
spot_check(test_profile, label="test")


================ UNIQUENESS CHECK: train ================
Customers: 12,500 | Unique entities used: 10,276
Entities reused: 1,743

================ INCOME MATCH CHECK: train ================
Income-known customers: 12,500
Median relative income error: 0.0001%  (should be ~0%, exact rescale applied)

================ TXN VOLUME BY BRACKET CHECK: train ================
                 count       mean       std   min        25%        50%  \
Income_Bracket                                                            
High            1591.0  59.918552  2.828280  55.0  57.500000  59.916667   
Low             4185.0  26.484448  0.864980  25.0  25.750000  26.500000   
Middle          6724.0  40.967170  1.739264  38.0  39.416667  41.000000   

                      75%   max  
Income_Bracket                   
High            62.291667  65.0  
Low             27.250000  28.0  
Middle          42.500000  44.0  

Reference ranges: {'Low': (25, 28), 'Middle': (38, 44), 'High': (55, 65)}

=======